# Bronze — physical_lojas

Desenvolvido por: Luiz Henrique Portácio

Este notebook lê o arquivo `physical_lojas.csv` da camada Raw e
grava os dados na Bronze em Delta.

**Regras aplicadas (planilha Squad 3 — regras técnicas 1-3):**
- Preservar os dados brutos como string.
- Adicionar auditoria com `bronze_ingested_at` e `bronze_source_file`.
- Gravar em modo append incremental.

**Regras técnicas (tratadas na Silver):**
1. `id_loja` não pode ser nulo nem duplicado (PK).
2. `cnpj` deve ter 14 dígitos numéricos e ser único.
3. `estado_loja` deve ser UF válida (2 letras).

**KPIs de negócio (gerados na Gold):**
4. Receita total por loja por ano.
5. Crescimento YoY da receita por loja.
6. Número de transações por loja por mês.
7. Enriquecimento IBGE (população por cidade).
8. Ticket médio por loja por semestre.

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Define imports, caminhos e parâmetros da Bronze de physical_lojas.

from pyspark.sql.functions import col, count, when, current_timestamp, lit

SOURCE_FILE = "physical_lojas.csv"
SOURCE_PATH = f"{RAW_BATCH_PATH}{SOURCE_FILE}"

BRONZE_TABLE = "physical_lojas"
BRONZE_PATH  = f"{BRONZE_BASE_PATH}{BRONZE_TABLE}"

CSV_OPTIONS = {
    "header": "true",
    "inferSchema": "false",
}

EXPECTED_COLUMNS = [
    "id_loja",
    "nome_loja",
    "cnpj",
    "cidade_loja",
    "estado_loja",
    "peso_vendas",
]

KEY_COLUMNS = ["id_loja"]

BRONZE_WRITE_MODE = "overwrite"

adls_options = get_adls_options()

print("Notebook configurado.")
print(f"Origem Raw  : {SOURCE_PATH}")
print(f"Destino Bronze: {BRONZE_PATH}")
print(f"Modo de escrita: {BRONZE_WRITE_MODE}")

In [0]:
# ── Controle de arquivo já ingerido ─────────────────────────────────────────
# Verifica se o SOURCE_FILE já foi processado na Bronze.
# Impede duplicação em execuções incrementais (pipeline toda segunda às 5h).
try:
    df_bronze_existente = read_delta(BRONZE_PATH, adls_options)
    arquivos_ja_ingeridos = set([
        row["bronze_source_file"]
        for row in df_bronze_existente
        .select("bronze_source_file")
        .distinct()
        .collect()
    ])
    if SOURCE_FILE in arquivos_ja_ingeridos:
        print(f"[SKIP] Arquivo '{SOURCE_FILE}' já foi ingerido na Bronze.")
        print(f"       Para reingerir, use BRONZE_WRITE_MODE = 'overwrite'.")
        dbutils.notebook.exit(f"SKIP: {SOURCE_FILE} já ingerido.")
    else:
        print(f"[OK] Arquivo '{SOURCE_FILE}' ainda não ingerido. Prosseguindo.")
except Exception:
    print(f"[OK] Bronze ainda não existe. Primeira ingestão de '{SOURCE_FILE}'.")

In [0]:
# Lê o CSV da Raw e valida se as colunas esperadas existem.

df_source = read_source_csv(
    spark=spark,
    source_path=SOURCE_PATH,
    adls_options=adls_options,
    csv_options=CSV_OPTIONS,
)

actual_columns  = df_source.columns
missing_columns = [c for c in EXPECTED_COLUMNS if c not in actual_columns]
extra_columns   = [c for c in actual_columns if c not in EXPECTED_COLUMNS]

if missing_columns:
    raise Exception(f"Colunas obrigatórias ausentes na origem: {missing_columns}")

if extra_columns:
    print(f"Atenção: colunas extras encontradas na origem: {extra_columns}")

total_source = df_source.count()

print("Validação inicial OK.")
print(f"Total de registros lidos da Raw: {total_source}")

df_source.printSchema()

In [0]:
# Cria a Bronze em memória com dados brutos e metadados de auditoria.

df_bronze = (
    df_source
    .select(
        *[col(c).cast("string").alias(c) for c in df_source.columns],
        col("_metadata.file_path").alias("bronze_source_file"),
    )
    .withColumn("bronze_ingested_at", current_timestamp())
)

display(df_bronze.limit(10))

In [0]:
# Proteção contra ingestão dupla — verifica se o arquivo já foi processado
try:
    df_bronze_existente = read_delta(BRONZE_PATH, adls_options)
    arquivos_ja_ingeridos = [
        r["bronze_source_file"] 
        for r in df_bronze_existente
            .select("bronze_source_file")
            .distinct()
            .collect()
    ]
    
    if SOURCE_PATH in arquivos_ja_ingeridos:
        print(f"⚠️ ATENÇÃO: O arquivo '{SOURCE_FILE}' já foi ingerido anteriormente.")
        print("Interrompa o notebook se isso não for esperado.")
        print("Se for uma atualização intencional, ignore este aviso.")
    else:
        print(f"[OK] Arquivo '{SOURCE_FILE}' ainda não ingerido. Prosseguindo.")
except Exception:
    print("[OK] Bronze ainda não existe. Primeira ingestão.")

In [0]:
# Grava a Bronze em Delta (append incremental).

(
    df_bronze
    .write
    .format("delta")
    .options(**adls_options)
    .option("mergeSchema", "true")
    .mode(BRONZE_WRITE_MODE)
    .save(BRONZE_PATH)
)

print(f"Bronze gravada com sucesso em Delta: {BRONZE_PATH}")
print(f"Modo de escrita utilizado: {BRONZE_WRITE_MODE}")
print(f"Total gravado na Bronze: {df_bronze.count()}")

In [0]:
# Valida volume e campos de auditoria após a escrita.

df_bronze_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(BRONZE_PATH)
)

total_bronze_saved = df_bronze_saved.count()
total_gravado      = df_bronze.count()

print(f"Total preparado para gravação : {total_gravado}")
print(f"Total Bronze gravada          : {total_bronze_saved}")

df_validacao = df_bronze_saved.select(
    count("*").alias("total_linhas"),
    count(when(col("id_loja").isNull(),          True)).alias("id_loja_nulo"),
    count(when(col("bronze_ingested_at").isNull(), True)).alias("bronze_ingested_at_nulo"),
    count(when(col("bronze_source_file").isNull(), True)).alias("bronze_source_file_nulo"),
)

display(df_validacao)

validacao = df_validacao.collect()[0]

if validacao["bronze_ingested_at_nulo"] > 0:
    raise Exception("Erro: existem registros sem bronze_ingested_at.")

if validacao["bronze_source_file_nulo"] > 0:
    raise Exception("Erro: existem registros sem bronze_source_file.")

print("Validação final da Bronze OK.")